# 03 Matching Comparison — Соусы

Цель: сравнить первые pairwise baselines на вручную размеченном gold-set и не смешивать калибровку с финальной оценкой.

- A: rule-based fuzzy + brand/weight/pack fusion.
- B: zero-shot multilingual bi-encoder, по умолчанию `intfloat/multilingual-e5-small`.
- `uncertain` остаётся для ручной разметки, но не входит в 3-class classification report.


## План

1. Загрузить `research/dedup/data/labeling_sauces.csv` и оставить только 3 оценочных класса.
2. Разделить размеченные пары на `dev` и `test` стратифицированно по label.
3. Посчитать score для A/B один раз.
4. Показать baseline-метрики на всём размеченном наборе как sanity-check.
5. Подобрать `threshold_high` на `dev`, затем оценить выбранный порог на held-out `test`.
6. Сохранить predictions/summary для `04_clustering_resolution.ipynb`.


In [1]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import math
import os
import sys
import time
from typing import Any

from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    BiEncoderMatcher,
    FusionConfig,
    RuleBasedMatcher,
    classification_report_df,
    confusion_matrix_df,
    decide_label,
)
from research.dedup.matchers.bi_encoder import BiEncoderConfig

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


In [2]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
LABELING_PATH = DATA_DIR / "labeling_sauces.csv"
PREDICTIONS_PATH = DATA_DIR / "matching_predictions_sauces.csv"
SUMMARY_PATH = DATA_DIR / "matching_summary_sauces.csv"
FALSE_MERGES_PATH = DATA_DIR / "matching_false_merges_sauces.csv"

EVAL_LABELS = ["exact_duplicate", "same_product_different_pack", "different_product"]
MERGE_LIKE_LABELS = {"exact_duplicate", "same_product_different_pack"}

RUN_BI_ENCODER = os.environ.get("DEDUP_RUN_BI_ENCODER", "1") == "1"
BI_ENCODER_MODEL = os.environ.get("DEDUP_BI_ENCODER_MODEL", "intfloat/multilingual-e5-small")
RANDOM_STATE = int(os.environ.get("DEDUP_EVAL_RANDOM_STATE", "42"))
DEV_FRACTION = float(os.environ.get("DEDUP_EVAL_DEV_FRACTION", "0.60"))
TARGET_EXACT_PRECISION = float(os.environ.get("DEDUP_TARGET_EXACT_PRECISION", "0.85"))
THRESHOLD_GRID = [round(value / 100, 2) for value in range(60, 101)]

print(f"Labeling path: {LABELING_PATH}")
print(f"Run bi-encoder: {RUN_BI_ENCODER}")
print(f"Bi-encoder model: {BI_ENCODER_MODEL}")
print(f"Dev fraction: {DEV_FRACTION:.0%}; random_state={RANDOM_STATE}")
print(f"Target exact_duplicate precision for calibration: {TARGET_EXACT_PRECISION:.0%}")


Labeling path: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/labeling_sauces.csv
Run bi-encoder: True
Bi-encoder model: intfloat/multilingual-e5-small
Dev fraction: 60%; random_state=42
Target exact_duplicate precision for calibration: 85%


In [3]:
def load_labeled_pairs(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        status = pd.DataFrame([
            {
                "status": "missing_labeling_file",
                "message": f"Файл {path} пока не найден. Выполните 02_labeling_dataset.ipynb и заполните label.",
                "rows_total": 0,
                "rows_usable_for_metrics": 0,
                "rows_ignored_non_3class": 0,
            }
        ])
        return pd.DataFrame(), status

    frame = pd.read_csv(path)
    if "label" not in frame.columns:
        status = pd.DataFrame([
            {
                "status": "missing_label_column",
                "message": "В файле нет колонки label. Перегенерируйте labeling dataset из notebook-2.",
                "rows_total": len(frame),
                "rows_usable_for_metrics": 0,
                "rows_ignored_non_3class": 0,
            }
        ])
        return pd.DataFrame(), status

    labels = frame["label"].fillna("").astype(str).str.strip()
    labeled = frame[labels.isin(EVAL_LABELS)].copy()
    labeled["label"] = labels[labels.isin(EVAL_LABELS)].to_numpy()
    ignored_count = int(labels.ne("").sum() - len(labeled))
    status_name = "ready" if not labeled.empty else "empty_or_not_reviewed_yet"
    message = (
        "Gold-set готов для метрик."
        if not labeled.empty
        else "3-class labels пока не заполнены: метрики ниже будут заглушками, notebook не падает."
    )
    status = pd.DataFrame([
        {
            "status": status_name,
            "message": message,
            "rows_total": len(frame),
            "rows_usable_for_metrics": len(labeled),
            "rows_ignored_non_3class": ignored_count,
        }
    ])
    return labeled.reset_index(drop=True), status


def add_stratified_eval_split(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.assign(eval_split=pd.Series(dtype="string"))
    parts: list[pd.DataFrame] = []
    for _, group in frame.groupby("label", sort=False):
        shuffled = group.sample(frac=1.0, random_state=RANDOM_STATE)
        if len(shuffled) == 1:
            dev = shuffled.copy()
            dev["eval_split"] = "dev"
            parts.append(dev)
            continue
        dev_count = int(round(len(shuffled) * DEV_FRACTION))
        dev_count = min(max(1, dev_count), len(shuffled) - 1)
        dev = shuffled.iloc[:dev_count].copy()
        test = shuffled.iloc[dev_count:].copy()
        dev["eval_split"] = "dev"
        test["eval_split"] = "test"
        parts.extend([dev, test])
    return pd.concat(parts).sort_index().reset_index(drop=True)


labeled_pairs, labeling_status = load_labeled_pairs(LABELING_PATH)
labeled_pairs = add_stratified_eval_split(labeled_pairs)

display(labeling_status)
if labeled_pairs.empty:
    display(pd.DataFrame(columns=["label", "pairs"]))
else:
    display(labeled_pairs["label"].value_counts().rename_axis("label").reset_index(name="pairs"))
    display(pd.crosstab(labeled_pairs["eval_split"], labeled_pairs["label"]))


,status,message,rows_total,rows_usable_for_metrics,rows_ignored_non_3class
0,ready,Gold-set готов для метрик.,400,379,21


,label,pairs
0,different_product,231
1,same_product_different_pack,76
2,exact_duplicate,72


label,different_product,exact_duplicate,same_product_different_pack
eval_split,,,
dev,139,43,46
test,92,29,30


In [4]:
matchers = [RuleBasedMatcher()]
if RUN_BI_ENCODER:
    matchers.append(BiEncoderMatcher(BiEncoderConfig(model_name=BI_ENCODER_MODEL)))

method_status = []
for matcher in matchers:
    status = matcher.status()
    method_status.append({"method": matcher.name, "available": status.available, "status": status.message})

method_status_df = pd.DataFrame(method_status)
display(method_status_df)


,method,available,status
0,rule_based_fuzzy,True,rapidfuzz available
1,bi_encoder_zero_shot,True,sentence-transformers available; model loads lazily


In [5]:
def _base_fusion_config(matcher: Any) -> FusionConfig:
    config = getattr(matcher, "config", None)
    return getattr(config, "fusion", FusionConfig())


def _fallback_label(matcher: Any) -> str:
    config = getattr(matcher, "config", None)
    return getattr(config, "uncertain_fallback_label", "different_product")


def _predict_from_score(matcher: Any, row: pd.Series, score: float, *, threshold_high: float | None = None) -> str:
    if math.isnan(score):
        return "different_product"
    fusion_config = _base_fusion_config(matcher)
    if threshold_high is not None:
        fusion_config = replace(fusion_config, threshold_high=threshold_high)
    label = decide_label(row, score, fusion_config)
    if label == "uncertain":
        return _fallback_label(matcher)
    return label


def _score_matcher(matcher: Any, pairs: pd.DataFrame) -> tuple[list[float], str]:
    if pairs.empty:
        return [], "skipped_empty_gold_set"
    row_objects = [row for _, row in pairs.iterrows()]
    if isinstance(matcher, BiEncoderMatcher):
        scores = matcher.score_batch(row_objects)
        if all(math.isnan(score) for score in scores):
            return scores, matcher.status().message
        return scores, "ready"
    return [matcher.score(row) for row in row_objects], "ready"


def _summarize_predictions(method: str, frame: pd.DataFrame, *, mode: str, threshold_high: float | None) -> dict[str, object]:
    report = classification_report_df(frame["label"], frame["predicted_label"], labels=EVAL_LABELS)
    macro = report[["precision", "recall", "f1"]].mean()
    exact_precision = float(report.loc[report["label"].eq("exact_duplicate"), "precision"].iloc[0])
    false_merges = frame[frame["false_merge"]]
    return {
        "method": method,
        "mode": mode,
        "eval_split": frame["eval_split"].iloc[0] if frame["eval_split"].nunique() == 1 else "all",
        "threshold_high": threshold_high,
        "pairs": len(frame),
        "macro_precision": float(macro["precision"]),
        "macro_recall": float(macro["recall"]),
        "macro_f1": float(macro["f1"]),
        "exact_duplicate_precision": exact_precision,
        "false_merge_count": int(len(false_merges)),
        "false_merge_rate": float(len(false_merges) / len(frame)) if len(frame) else 0.0,
    }


def _predict_frame(matcher: Any, frame: pd.DataFrame, scores: list[float], *, threshold_high: float | None, mode: str) -> pd.DataFrame:
    output = frame.copy()
    output["method"] = matcher.name
    output["score"] = scores
    output["threshold_high"] = threshold_high
    output["mode"] = mode
    output["predicted_label"] = [
        _predict_from_score(matcher, row, score, threshold_high=threshold_high)
        for (_, row), score in zip(output.iterrows(), scores, strict=False)
    ]
    output["false_merge"] = output["predicted_label"].isin(MERGE_LIKE_LABELS) & output["label"].eq("different_product")
    return output


In [6]:
scored_methods: dict[str, dict[str, object]] = {}
skipped_methods: list[dict[str, str]] = []

if labeled_pairs.empty:
    display(pd.DataFrame([{"method": "not_available_yet", "status": labeling_status.loc[0, "status"], "seconds": 0.0}]))
else:
    scoring_rows = []
    for matcher in matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, labeled_pairs)
        elapsed = time.perf_counter() - started
        scoring_rows.append({"method": matcher.name, "status": status, "seconds": round(elapsed, 3)})
        if status != "ready":
            skipped_methods.append({"method": matcher.name, "status": status})
            continue
        scored_methods[matcher.name] = {"matcher": matcher, "scores": scores, "seconds": elapsed}
    display(pd.DataFrame(scoring_rows))
    if skipped_methods:
        display(pd.DataFrame(skipped_methods))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4771.49it/s]

,method,status,seconds
0,rule_based_fuzzy,ready,0.036
1,bi_encoder_zero_shot,ready,18.903


In [7]:
default_outputs: dict[str, pd.DataFrame] = {}
default_reports: list[pd.DataFrame] = []
default_summary_rows: list[dict[str, object]] = []

for method, payload in scored_methods.items():
    matcher = payload["matcher"]
    scores = payload["scores"]
    output = _predict_frame(
        matcher,
        labeled_pairs,
        scores,
        threshold_high=_base_fusion_config(matcher).threshold_high,
        mode="default_full_gold_set_sanity",
    )
    default_outputs[method] = output
    default_summary_rows.append(
        _summarize_predictions(
            method,
            output,
            mode="default_full_gold_set_sanity",
            threshold_high=_base_fusion_config(matcher).threshold_high,
        )
    )
    report = classification_report_df(output["label"], output["predicted_label"], labels=EVAL_LABELS)
    report.insert(0, "method", method)
    report.insert(1, "mode", "default_full_gold_set_sanity")
    default_reports.append(report)

if default_summary_rows:
    default_summary_df = pd.DataFrame(default_summary_rows).sort_values("macro_f1", ascending=False)
    default_report_df = pd.concat(default_reports, ignore_index=True)
    display(default_summary_df)
    display(default_report_df)
else:
    display(pd.DataFrame([{"method": "not_available_yet", "mode": "default_full_gold_set_sanity", "pairs": 0}]))


,method,mode,eval_split,threshold_high,pairs,macro_precision,macro_recall,macro_f1,exact_duplicate_precision,false_merge_count,false_merge_rate
0,rule_based_fuzzy,default_full_gold_set_sanity,all,0.82,379,0.591363,0.648863,0.606437,0.425532,89,0.234828
1,bi_encoder_zero_shot,default_full_gold_set_sanity,all,0.86,379,0.501348,0.582786,0.391180,0.262097,215,0.567282


,method,mode,label,precision,recall,f1,support
0,rule_based_fuzzy,default_full_gold_set_sanity,exact_duplicate,0.425532,0.555556,0.481928,72
1,rule_based_fuzzy,default_full_gold_set_sanity,same_product_different_pack,0.546296,0.776316,0.641304,76
2,rule_based_fuzzy,default_full_gold_set_sanity,different_product,0.802260,0.614719,0.696078,231
3,bi_encoder_zero_shot,default_full_gold_set_sanity,exact_duplicate,0.262097,0.902778,0.406250,72
4,bi_encoder_zero_shot,default_full_gold_set_sanity,same_product_different_pack,0.546296,0.776316,0.641304,76
5,bi_encoder_zero_shot,default_full_gold_set_sanity,different_product,0.695652,0.069264,0.125984,231


In [8]:
def _threshold_scores_for_split(method: str, payload: dict[str, object], split_name: str) -> tuple[pd.DataFrame, list[float]]:
    split_mask = labeled_pairs["eval_split"].eq(split_name)
    split_frame = labeled_pairs.loc[split_mask].copy()
    split_scores = [score for score, keep in zip(payload["scores"], split_mask, strict=False) if keep]
    return split_frame, split_scores


calibration_rows: list[dict[str, object]] = []
calibrated_outputs: list[pd.DataFrame] = []
calibrated_reports: list[pd.DataFrame] = []
calibrated_summary_rows: list[dict[str, object]] = []

for method, payload in scored_methods.items():
    matcher = payload["matcher"]
    dev_frame, dev_scores = _threshold_scores_for_split(method, payload, "dev")
    if dev_frame.empty:
        continue

    grid_rows = []
    for threshold_high in THRESHOLD_GRID:
        dev_pred = _predict_frame(matcher, dev_frame, dev_scores, threshold_high=threshold_high, mode="calibration_dev")
        grid_rows.append(_summarize_predictions(method, dev_pred, mode="calibration_dev", threshold_high=threshold_high))
    grid = pd.DataFrame(grid_rows)
    target_met = grid[grid["exact_duplicate_precision"].ge(TARGET_EXACT_PRECISION)]
    selection_pool = target_met if not target_met.empty else grid
    selected = selection_pool.sort_values(
        ["macro_f1", "exact_duplicate_precision", "false_merge_rate"],
        ascending=[False, False, True],
    ).iloc[0]
    selected_threshold = float(selected["threshold_high"])
    calibration_rows.append(
        {
            "method": method,
            "selected_threshold_high": selected_threshold,
            "target_exact_precision": TARGET_EXACT_PRECISION,
            "target_met_on_dev": bool(selected["exact_duplicate_precision"] >= TARGET_EXACT_PRECISION),
            "dev_macro_f1": float(selected["macro_f1"]),
            "dev_exact_duplicate_precision": float(selected["exact_duplicate_precision"]),
            "dev_false_merge_rate": float(selected["false_merge_rate"]),
        }
    )

    all_pred = _predict_frame(matcher, labeled_pairs, payload["scores"], threshold_high=selected_threshold, mode="calibrated")
    calibrated_outputs.append(all_pred)
    for split_name in ["dev", "test"]:
        split_pred = all_pred[all_pred["eval_split"].eq(split_name)].copy()
        if split_pred.empty:
            continue
        calibrated_summary_rows.append(
            _summarize_predictions(method, split_pred, mode="calibrated", threshold_high=selected_threshold)
        )
        report = classification_report_df(split_pred["label"], split_pred["predicted_label"], labels=EVAL_LABELS)
        report.insert(0, "method", method)
        report.insert(1, "mode", "calibrated")
        report.insert(2, "eval_split", split_name)
        report.insert(3, "threshold_high", selected_threshold)
        calibrated_reports.append(report)

calibration_df = pd.DataFrame(calibration_rows)
calibrated_summary_df = pd.DataFrame(calibrated_summary_rows)
calibrated_report_df = pd.concat(calibrated_reports, ignore_index=True) if calibrated_reports else pd.DataFrame()
all_calibrated_predictions = pd.concat(calibrated_outputs, ignore_index=True) if calibrated_outputs else pd.DataFrame()

if not calibration_df.empty:
    display(calibration_df)
    display(calibrated_summary_df.sort_values(["eval_split", "macro_f1"], ascending=[True, False]))
    display(calibrated_report_df)
else:
    display(pd.DataFrame([{"status": "no_methods_available_for_calibration"}]))


,method,selected_threshold_high,target_exact_precision,target_met_on_dev,dev_macro_f1,dev_exact_duplicate_precision,dev_false_merge_rate
0,rule_based_fuzzy,0.86,0.85,False,0.618742,0.517241,0.127193
1,bi_encoder_zero_shot,1.00,0.85,True,0.511972,1.000000,0.105263


,method,mode,eval_split,threshold_high,pairs,macro_precision,macro_recall,macro_f1,exact_duplicate_precision,false_merge_count,false_merge_rate
0,rule_based_fuzzy,calibrated,dev,0.86,228,0.627863,0.626445,0.618742,0.517241,29,0.127193
2,bi_encoder_zero_shot,calibrated,dev,1.00,228,0.758277,0.545412,0.511972,1.000000,24,0.105263
1,rule_based_fuzzy,calibrated,test,0.86,151,0.619812,0.636099,0.606322,0.578947,28,0.185430
3,bi_encoder_zero_shot,calibrated,test,1.00,151,0.730000,0.539272,0.470139,1.000000,23,0.152318


,method,mode,eval_split,threshold_high,label,precision,recall,f1,support
0,rule_based_fuzzy,calibrated,dev,0.86,exact_duplicate,0.517241,0.348837,0.416667,43
1,rule_based_fuzzy,calibrated,dev,0.86,same_product_different_pack,0.586207,0.739130,0.653846,46
2,rule_based_fuzzy,calibrated,dev,0.86,different_product,0.780142,0.791367,0.785714,139
3,rule_based_fuzzy,calibrated,test,0.86,exact_duplicate,0.578947,0.379310,0.458333,29
4,rule_based_fuzzy,calibrated,test,0.86,same_product_different_pack,0.500000,0.833333,0.625000,30
5,rule_based_fuzzy,calibrated,test,0.86,different_product,0.780488,0.695652,0.735632,92
6,bi_encoder_zero_shot,calibrated,dev,1.00,exact_duplicate,1.000000,0.069767,0.130435,43
7,bi_encoder_zero_shot,calibrated,dev,1.00,same_product_different_pack,0.586207,0.739130,0.653846,46
8,bi_encoder_zero_shot,calibrated,dev,1.00,different_product,0.688623,0.827338,0.751634,139
9,bi_encoder_zero_shot,calibrated,test,1.00,exact_duplicate,1.000000,0.034483,0.066667,29


In [9]:
if all_calibrated_predictions.empty:
    print("Confusion matrices и false-merge examples появятся после запуска хотя бы одного метода на размеченных парах.")
else:
    for method in all_calibrated_predictions["method"].drop_duplicates():
        for split_name in ["dev", "test"]:
            part = all_calibrated_predictions[
                all_calibrated_predictions["method"].eq(method)
                & all_calibrated_predictions["eval_split"].eq(split_name)
            ]
            if part.empty:
                continue
            print(f"Confusion matrix: {method} / calibrated / {split_name}")
            display(confusion_matrix_df(part["label"], part["predicted_label"], labels=EVAL_LABELS))

    false_merge_examples = all_calibrated_predictions[
        all_calibrated_predictions["eval_split"].eq("test") & all_calibrated_predictions["false_merge"]
    ].copy()
    if false_merge_examples.empty:
        print("На held-out test false-merge примеров нет.")
    else:
        print(f"False-merge examples on held-out test: {len(false_merge_examples)}")
        columns = [
            "method",
            "label",
            "predicted_label",
            "score",
            "threshold_high",
            "title_a",
            "title_b",
            "brand_a",
            "brand_b",
            "unit_amount_a",
            "unit_amount_b",
            "total_amount_a",
            "total_amount_b",
            "multipack_count_a",
            "multipack_count_b",
        ]
        display(false_merge_examples[[column for column in columns if column in false_merge_examples.columns]].head(20))


Confusion matrix: rule_based_fuzzy / calibrated / dev


predicted_label,exact_duplicate,same_product_different_pack,different_product
true_label,,,
exact_duplicate,15,0,28
same_product_different_pack,9,34,3
different_product,5,24,110


Confusion matrix: rule_based_fuzzy / calibrated / test


predicted_label,exact_duplicate,same_product_different_pack,different_product
true_label,,,
exact_duplicate,11,2,16
same_product_different_pack,3,25,2
different_product,5,23,64


Confusion matrix: bi_encoder_zero_shot / calibrated / dev


predicted_label,exact_duplicate,same_product_different_pack,different_product
true_label,,,
exact_duplicate,3,0,40
same_product_different_pack,0,34,12
different_product,0,24,115


Confusion matrix: bi_encoder_zero_shot / calibrated / test


predicted_label,exact_duplicate,same_product_different_pack,different_product
true_label,,,
exact_duplicate,1,2,26
same_product_different_pack,0,25,5
different_product,0,23,69


False-merge examples on held-out test: 51


,method,label,predicted_label,score,threshold_high,title_a,title_b,brand_a,brand_b,unit_amount_a,unit_amount_b,total_amount_a,total_amount_b,multipack_count_a,multipack_count_b
72,rule_based_fuzzy,different_product,same_product_different_pack,0.604545,0.86,Соус Пад-Тай для лапши 5 шт х 80гр,соус Том Ям основа для супа 80гр,sen soy,sen soy,0.080,0.080,0.400,0.080,5.0,1.0
76,rule_based_fuzzy,different_product,same_product_different_pack,0.668519,0.86,Корейская заправка для салата овощного из спаржи 3шт*60г,Заправка для салатов корейская для острой морковки 2/60г,чим-чим,чим-чим,0.060,0.060,0.180,0.060,3.0,1.0
102,rule_based_fuzzy,different_product,same_product_different_pack,0.627000,0.86,"Соус Томатный низкокалорийный без сахара Bombbar, 2шт х 240г","Бомбар, Низкокалорийный соус 240 г, Барбекю",bombbar,bombbar,0.240,0.240,0.480,0.240,2.0,1.0
103,rule_based_fuzzy,different_product,same_product_different_pack,0.513514,0.86,СТЕБЕЛЬ БАМБУКА Соус к мясу ПЭТ 280 гр 2 шт,Соус чили острый 280 гр 1 штука,стебель бамбука,стебель бамбука,0.280,0.280,0.560,0.280,2.0,1.0
108,rule_based_fuzzy,different_product,exact_duplicate,0.891358,0.86,Соевый соус SanBonsai Терияки Stir Fry WOK с/б 275 гр,Соевый соус SanBonsai Терияки Stir Fry 300г,sanbonsai,sanbonsai,0.275,0.300,0.275,0.300,1.0,1.0
110,rule_based_fuzzy,different_product,exact_duplicate,0.948450,0.86,"Уксус Monini Aceto Balsamico di Modena IGP Винный бальзамический, 500мл","Уксус Monini Aceto Balsamico винный бальзамический 6%, 250 мл",monini,monini,0.500,0.250,0.500,0.250,1.0,1.0
135,rule_based_fuzzy,different_product,same_product_different_pack,0.596078,0.86,"Соус Calve Баварский медово-горчичный, 230 г 4 шт","Соус сливочно-чесночный, 230 г Calve",calve,calve,0.230,0.230,0.920,0.230,4.0,1.0
158,rule_based_fuzzy,different_product,exact_duplicate,0.876190,0.86,"Низкокалорийный соус Mr.Djemius ZERO ""Сладкий чили"" 330г","Низкокалорийный соус Mr.Djemius ZERO ""Сацебели"" 330гр",mr. djemius zero,mr. djemius zero,0.330,0.330,0.330,0.330,1.0,1.0
160,rule_based_fuzzy,different_product,exact_duplicate,0.876712,0.86,Соус Ореховый и Кимчи 470 мл 2 шт,Соус Ореховый и Сладкий чили 470 мл 2 шт,tamaki,tamaki,0.470,0.470,0.940,0.940,2.0,2.0
163,rule_based_fuzzy,different_product,exact_duplicate,0.865823,0.86,Низкокалорийный соус без сахара Сальса 330г,"Соус Сладкий чили низкокалорийный, без сахара 330г",mr.djemius zero,mr. djemius zero,0.330,0.330,0.330,0.330,1.0,1.0


In [10]:
summary_frames = []
if default_summary_rows:
    summary_frames.append(pd.DataFrame(default_summary_rows))
if calibrated_summary_rows:
    summary_frames.append(pd.DataFrame(calibrated_summary_rows))

if summary_frames:
    summary_export = pd.concat(summary_frames, ignore_index=True)
    summary_export.to_csv(SUMMARY_PATH, index=False)
    print(f"Saved summary: {SUMMARY_PATH} ({len(summary_export)} rows)")
else:
    summary_export = pd.DataFrame()
    print("Summary export skipped: no methods available.")

if not all_calibrated_predictions.empty:
    all_calibrated_predictions.to_csv(PREDICTIONS_PATH, index=False)
    print(f"Saved predictions: {PREDICTIONS_PATH} ({len(all_calibrated_predictions)} rows)")
    false_merges_export = all_calibrated_predictions[all_calibrated_predictions["false_merge"]].copy()
    false_merges_export.to_csv(FALSE_MERGES_PATH, index=False)
    print(f"Saved false merges: {FALSE_MERGES_PATH} ({len(false_merges_export)} rows)")
else:
    print("Prediction export skipped: no calibrated outputs.")


Saved summary: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_summary_sauces.csv (6 rows)
Saved predictions: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_predictions_sauces.csv (758 rows)
Saved false merges: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_false_merges_sauces.csv (104 rows)


## Следующие шаги

- Использовать `matching_predictions_sauces.csv` в `04_clustering_resolution.ipynb` как вход для графовой сборки family/pack groups.
- Не считать текущий bi-encoder финальным результатом: zero-shot score слишком часто мержит похожие, но разные вкусы/типы.
- Для следующего качества нужен либо более строгий fusion, либо cross-encoder/reranker на размеченных hard negatives.
